# Classificador de uso das edificações: EDA e avaliação

Análise exploratória dos dados que alimentam o classificador e leitura das métricas do modelo treinado.

- **Treino reprodutível:** `ai-model/scripts/treinar_uso.py` (semente 42). Este notebook não treina; ele lê a mesma base do banco e o `ai-model/models/modelo_uso.json` gerado pelo script.
- **Para rodar:** MariaDB no ar com os dados importados (ver README), `.env` configurado e `pip install notebook`; abra este arquivo com o Jupyter na raiz do repositório.

In [1]:
import json, sys
from collections import Counter
from pathlib import Path

import numpy as np

RAIZ = Path.cwd() if (Path.cwd() / 'backend').exists() else Path.cwd().parents[1]
sys.path.insert(0, str(RAIZ))

from backend import db
from backend.ia.dados import de_linhas
from backend.ia.features import CLASSES, mascara_previsao, mascara_treino, montar_features

## 1. A base: quantos prédios têm uso conhecido

In [2]:
base = de_linhas(db.fetch_buildings_for_model())
treino = mascara_treino(base.tipos, base.origens)
prever = mascara_previsao(base.tipos)
n = len(base.tipos)
print(f'Prédios: {n}')
print(f'Rotulados pelo CNEFE (exemplos de treino): {treino.sum()} ({treino.mean():.1%})')
print(f'Sem uso conhecido (recebem previsão): {prever.sum()} ({prever.mean():.1%})')

contagem = Counter(t for t, ok in zip(base.tipos, treino) if ok)
print('\nClasse          exemplos   fatia')
for c in CLASSES:
    print(f'{c:15s} {contagem[c]:8d}  {contagem[c] / treino.sum():6.1%}')

Prédios: 55255
Rotulados pelo CNEFE (exemplos de treino): 23115 (41.8%)
Sem uso conhecido (recebem previsão): 32071 (58.0%)

Classe          exemplos   fatia
residencial        21368   92.4%
comercial           1522    6.6%
servico               37    0.2%
religioso             79    0.3%
institucional         53    0.2%
hospedagem            15    0.1%
agropecuario          41    0.2%


**Leitura.** Mais da metade dos prédios (58%) não tem uso conhecido: é o problema que o classificador ataca. Entre os rotulados, a base é muito desbalanceada: residencial é 92,4% e as cinco classes raras somam menos de 1% (15 a 79 exemplos cada). Isso define a avaliação: a acurácia engana, porque um palpite fixo em "residencial" já acerta 92,4%, então a métrica principal é o F1 macro, que dá o mesmo peso a cada classe.

## 2. Forma e fonte: o que separa as classes e o que muda nos prédios sem endereço

In [3]:
X, nomes = montar_features(base.geoms, base.fontes, base.tipos, treino)
area = X[:, nomes.index('area_m2')]
tipos = np.array(base.tipos, dtype=object)

print('Área do footprint (m²)       mediana   p90    n')
for c in CLASSES:
    a = area[treino & (tipos == c)]
    print(f'{c:25s} {np.median(a):8.0f} {np.percentile(a, 90):6.0f} {len(a):6d}')
a = area[prever]
print(f"{'desconhecido (sem endereço)':25s} {np.median(a):8.0f} {np.percentile(a, 90):6.0f} {len(a):6d}")

fontes = np.array(base.fontes, dtype=object)
print('\nFonte do footprint            rotulados   sem endereço')
for f in sorted(set(base.fontes)):
    print(f'{f:28s} {np.mean(fontes[treino] == f):9.1%} {np.mean(fontes[prever] == f):13.1%}')

Área do footprint (m²)       mediana   p90    n
residencial                    106    234  21368
comercial                      106    364   1522
servico                        214    689     37
religioso                      103    272     79
institucional                  159    769     53
hospedagem                     208    469     15
agropecuario                    67    128     41
desconhecido (sem endereço)       58    168  32071

Fonte do footprint            rotulados   sem endereço
Google Open Buildings            82.8%         79.4%
Microsoft ML Buildings           13.5%         17.7%
OpenStreetMap                     3.7%          2.9%


**Leitura.**

- **Residencial e comercial têm a mesma área mediana (106 m²).** Só a cauda difere (p90 de 234 m² contra 364 m²): comércios grandes são separáveis pela forma, os pequenos não. Isso explica o recall baixo de comercial no modelo e por que as features de vizinhança (uso dos vizinhos, distância ao comercial mais próximo) entram no modelo.
- **Os prédios sem endereço são menores** (mediana de 58 m² contra 106 m² dos residenciais rotulados) e vêm um pouco mais do Microsoft ML Buildings. É a evidência do viés de seleção: a população que recebe a previsão não é igual à de treino. Parte desses prédios deve ser anexo, garagem ou galpão, usos que nem existem como classe no CNEFE. A validação espacial mede só parte desse efeito, e esse limite aparece na seção "Sobre o modelo" do sistema.

## 3. Avaliação do modelo (gerada por `treinar_uso.py`)

Validação cruzada em 5 folds: estratificada (sorteio de prédios) e espacial (células de 500 m inteiras no teste). A métrica principal é o F1 macro, porque a base é desbalanceada.

In [4]:
info = json.loads((RAIZ / 'ai-model' / 'models' / 'modelo_uso.json').read_text(encoding='utf-8'))
m = info['metricas']
print('Modelo                 F1 estratif.  F1 espacial  acurácia espacial')
for nome, r in m['modelos'].items():
    print(f"{nome:22s} {r['estratificada']['f1_macro']:12.3f} {r['espacial']['f1_macro']:12.3f} {r['espacial']['acuracia']:18.3f}")
print('\nAjuste de hiperparâmetros:', *[f"{g['params']} -> {g['f1_macro']:.3f}" for g in m['grade']], sep='\n  ')

final = m['modelo_final']
print(f"\nModelo final calibrado ({info['versao']}): F1 macro {final['f1_macro']:.3f}, acurácia {final['acuracia']:.3f}")
print('Classe          precisão recall   F1   suporte')
for c, v in final['por_classe'].items():
    print(f"{c:15s} {v['precisao']:8.2f} {v['recall']:6.2f} {v['f1']:5.2f} {v['suporte']:8d}")

classes = m['classes']
print('\nMatriz de confusão (linhas = oficial, colunas = previsto)')
print(' ' * 14 + ''.join(f'{c[:6]:>8s}' for c in classes))
for c, linha in zip(classes, final['matriz_confusao']):
    print(f'{c:14s}' + ''.join(f'{x:8d}' for x in linha))

Modelo                 F1 estratif.  F1 espacial  acurácia espacial
mais_frequente                0.137        0.137              0.924
regressao_logistica           0.139        0.121              0.403
random_forest                 0.167        0.143              0.925
gradient_boosting             0.194        0.135              0.541

Ajuste de hiperparâmetros:
  {'modelo': 'random_forest', 'n_estimators': 200, 'min_samples_leaf': 1} -> 0.143
  {'modelo': 'random_forest', 'n_estimators': 200, 'min_samples_leaf': 3} -> 0.167
  {'modelo': 'random_forest', 'n_estimators': 400, 'min_samples_leaf': 1} -> 0.143
  {'modelo': 'random_forest', 'n_estimators': 400, 'min_samples_leaf': 3} -> 0.166

Modelo final calibrado (uso-20261122-a82af781): F1 macro 0.155, acurácia 0.926
Classe          precisão recall   F1   suporte
residencial         0.93   1.00  0.96    21368
comercial           0.62   0.07  0.13     1522
servico             0.00   0.00  0.00       37
religioso           0.00   0.00 

**Leitura.**

- **Comparação com o baseline:** na validação espacial, que é a mais honesta para prédios sem endereço (áreas inteiras ficam fora do treino), o melhor modelo (Random Forest, `min_samples_leaf=3`) chega a F1 macro de 0,167, contra 0,137 da classe mais frequente. O ganho existe, mas é pequeno. O Gradient Boosting vai melhor na validação estratificada (0,194), mas cai na espacial (0,135): ele decora a vizinhança e não generaliza para áreas novas. Por isso a escolha foi pela espacial.
- **Ajuste:** folhas com pelo menos 3 amostras sobem o F1 de 0,143 para 0,167. Dobrar as árvores para 400 não muda o resultado.
- **Onde erra:** residencial tem F1 de 0,96. Comercial tem precisão de 0,62 e recall de 0,07: quando o modelo diz "comercial" costuma acertar, mas deixa passar quase todos (1.416 dos 1.522 viram residencial na matriz). As classes raras nunca são acertadas, porque com 15 a 79 exemplos o modelo não as aprende.
- **É bom o bastante para o uso proposto?** Para estimar o uso de prédios sem endereço e priorizar conferência de campo, sim, desde que como estimativa: uma previsão diferente de residencial é um sinal útil (precisão de 0,62 em comercial), e a confiança ajuda a ordenar. Não serve para afirmar o uso de um prédio específico, nem para contar as classes raras. Caminhos de melhoria: juntar as classes raras em "não residencial", mais rótulos (OSM `amenity`/`shop`) e um limiar de decisão por classe.
